# El barrido de ruido — la corrida que recorre los niveles

Este cuaderno **corre** el barrido y nada más: no arma ninguna tabla ni conclusión — eso vive en `Benchmark_Noise_Report_v1.ipynb`, que lee los `runs.jsonl` que esto deja y dibuja la curva de degradación.

Es una forma distinta de una campaña y no una campaña más chica: **una transferencia recorriendo cada nivel declarado**, contra seis transferencias en un nivel. Por eso escribe bajo `kind="curve"` — las dos pueden pararse en la misma tasa, y `runs.jsonl` se abre en `"w"`.

> **Los techos salen de la búsqueda en limpio y se mantienen fijos en los cinco niveles.** La curva es el coeficiente elegido sin contaminación aplicado con ella, que además es la situación práctica. Lo que eso cuesta es que una caída no se pueda atribuir: el término fallando y el coeficiente quedándose corto se ven igual. Eso lo separa `Benchmark_Noise_Diagnostic_Search_v1.ipynb`, que re-busca a su nivel y no gobierna este registro.
>
> **Se leen del registro, y no se busca nada acá.** Llamar a `harness.with_ceilings_in_force` era el reflejo obvio y es una trampa: cuando no existe `ceilings.json` esa función lanza la búsqueda COMPLETA — dos familias × seis transferencias × treinta trials a veinte épocas, unas nueve horas y media — sin decir que lo está haciendo y sin que nadie la haya autorizado. Un barrido a escala de ensayo no puede ser la puerta por la que entra la corrida larga.
>
> **Guarda pesos en dos de los cinco niveles y en ninguno de los otros tres,** y no lo decide este cuaderno: `campaign()` pregunta por `keeps_checkpoints`, que es verdadero exactamente en los niveles que el cuaderno latente dibuja. Los otros tres corren y escriben sus `runs.jsonl`, que es de donde sale la curva, y ni un peso.

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import time
from dataclasses import replace

from MIL_CREDA_Benchmark import config, harness

device = harness.resolve_device()
# La escala configurada decide si esto es un ensayo, y el ensayo decide DÓNDE
# escribe. `config.is_pilot_scale()` es la única lectura de esa regla --- las dos
# constantes que separan una escala de la otra son `EPOCHS` y `SEEDS` ---, y
# escribirla otra vez acá serían dos ortografías de lo mismo. El paso que corre
# este cuaderno se niega cuando esa lectura dice que la escala es la completa,
# porque sus `produces` nombran el árbol de ENSAYO y ninguno más.
ES_ENSAYO = config.is_pilot_scale()
# Y de qué árbol se LEE lo que dejó la búsqueda, que es otra pregunta. Las dos
# son la misma en el recorrido local ---en ensayo todo corre y consume lo del
# ensayo--- y se separan en el ENSAYO REMOTO: ahí el barrido corre a escala
# reducida, en el worker, bajo el `ceilings.json` que la búsqueda dejó a escala
# COMPLETA, que es el archivo que el barrido real va a abrir. Fuera de ese modo
# esto es `is_pilot_scale()` y no cambia nada.
ES_ENSAYO_ENTRADA = config.upstream_pilot_scale()

# Sin registro no corre: un barrido con techos vacíos mediría el ruido y la falta
# de coeficiente a la vez. Y el registro que se pide es el de la escala de
# ENTRADA, el mismo que se lee tres líneas más abajo: la guarda preguntaba si
# existía ALGUNO de los dos archivos mientras la lectura se llevaba el vigente,
# así que a escala de ensayo con un `ceilings.json` en disco la guarda decía «hay
# registro» y el barrido corría bajo los techos de la corrida COMPLETA sin
# pedirlos. Una pregunta y una lectura contra archivos distintos son dos mitades
# que pueden discrepar; contra el mismo archivo no pueden --- y por eso las dos
# se mueven con la MISMA coordenada, también cuando esa coordenada es la del
# ensayo remoto, donde leer lo completo es justamente el punto.
if harness.search_record(pilot=ES_ENSAYO_ENTRADA) is None:
    raise SystemExit(
        "no hay registro de techos a la escala que este barrido lee. Corré "
        "primero la búsqueda "
        "(`search-pilot` a escala de ensayo, o la completa con su "
        "autorización): un barrido sin techos mide dos cosas a la vez.")

# Los techos que YA estén en el registro DE ESTA ESCALA, UNA vez y arriba del
# bucle: leerlos por nivel daría los mismos números hoy y dejaría el barrido a
# merced de un registro que cambie a mitad de corrida.
base = replace(
    harness.Reduction(device=str(device), environment=harness.environment(),
                      pilot=ES_ENSAYO, kind="curve"),
    ceilings=config.ceilings_on_record(pilot=ES_ENSAYO_ENTRADA),
    ceilingsByTransfer=config.ceilings_by_transfer_on_record(
        pilot=ES_ENSAYO_ENTRADA))

print(harness.header(base))
print()
print("transferencia:", "{}->{}".format(*config.NOISE_TRANSFER))
print("niveles:", [f"{r:g}" for r in config.NOISE_LEVELS])
print("escala:", "ensayo" if ES_ENSAYO else "completa",
      "| escribe en", config.results_for(0.0, base.kind, base.pilot).parent)

## Lo que cuesta

Una corrida real, cronometrada, antes de comprometerse con los cinco niveles: un
estimado del costo es más barato que el costo, así que va primero. Por los cinco
niveles y no por uno — la rejilla de UNA transferencia corre entera una vez por
nivel, y un pronóstico de un solo nivel diría un quinto de lo que el cuaderno va
a gastar, que es peor que no pronosticar nada.

In [ ]:
from MIL_CREDA_Benchmark import bags

material = {rol: bags.build(codigo, config.DATA_CACHE, config.SEEDS[0])
            for rol, codigo in zip(("source", "target"), config.NOISE_TRANSFER)}
sonda = harness.run_one("G", config.NOISE_TRANSFER, config.SEEDS[0],
                        base, device, material)
por_corrida = sonda["seconds"]
forma = config.sizing()
# La rejilla de UNA transferencia, compuesta desde el aforo y no escrita: brazos
# por semillas, por los niveles declarados.
por_nivel = forma["arms"] * forma["seeds"]
rejilla = por_nivel * len(config.NOISE_LEVELS)
print(f"una corrida completa, {base.epochs} épocas: {por_corrida:.1f}s")
print(f"este barrido ({por_nivel} corridas x {len(config.NOISE_LEVELS)} niveles "
      f"= {rejilla} corridas): unos {rejilla * por_corrida / 60:.0f} min")
del material, sonda

## La corrida

Un nivel por pasada, todas bajo los mismos techos. Lo único que cambia entre una
y la siguiente es la tasa que lleva el material de entrenamiento.

In [ ]:
corridos = {}
for tasa in config.NOISE_LEVELS:
    pasada = replace(base, labelNoise=tasa)
    # De la reducción con la que se está por correr, no de la constante: las
    # coordenadas del primer nivel coinciden con `config.NOISE` por casualidad, y
    # esa casualidad es cómo se lee el árbol equivocado.
    raiz = config.results_for(pasada.labelNoise, pasada.kind, pasada.pilot)
    print(f"\nnivel ρ={tasa:g} → {raiz}")
    empezado = time.perf_counter()
    corridos[f"{tasa:g}"] = harness.campaign(
        pasada, device, transfers=[config.NOISE_TRANSFER])
    print(f"nivel ρ={tasa:g} terminado en "
          f"{(time.perf_counter() - empezado) / 60:.1f} min")

print()
print("la curva y su conclusión las arma Benchmark_Noise_Report_v1.ipynb")